# 不是缩放定律！

我们将玩一玩 Hugging Face 的 [Transformers](https://huggingface.co/docs/transformers/en/index) 和 [Datasets](https://huggingface.co/docs/datasets/en/index) 库。

问题是：给定计算预算，数据稀缺的影响是什么？

为了简化，我们加一些约束：
- 参数数量固定：你将微调 distilGPT2 模型。
- 我们把最大句子长度固定为 64 个 token，并假设计算预算允许你最多让一个句子前向和反向通过模型 100 次。


In [ ]:
import time
import datasets
from transformers import GPT2TokenizerFast, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling

下面的代码加载语料


In [ ]:
t = GPT2TokenizerFast.from_pretrained('distilgpt2')
t.pad_token = t.eos_token
dc = DataCollatorForLanguageModeling(tokenizer=t, mlm=False)
d0 = datasets.load_dataset("wikitext","wikitext-2-v1")
dval = d0['validation']
dtrain = d0['train']

下面的代码构建训练数据集和验证数据集


In [ ]:
slen = 64
def tokenize(element):
    outputs = t(element["text"], truncation=True, max_length=slen, return_overflowing_tokens=True, return_length=True)
    input_batch = []
    for length, input_ids in zip(outputs["length"], outputs["input_ids"]):
        if length == slen: input_batch.append(input_ids)
    return {"input_ids": input_batch}
dtrain = dtrain.map(tokenize, batched=True, remove_columns=dtrain.column_names)
dval = dval.map(tokenize, batched=True, remove_columns=dval.column_names)
print("training data",d0)

In [ ]:
dval = dval.select([i for i in range(10)])
print("validation data",dval)

下面是一个用 [`Trainer` 类](https://huggingface.co/docs/transformers/en/main_classes/trainer) 做微调的示例代码。


In [ ]:
d = dtrain.select([i for i in range(3)])
model = GPT2LMHeadModel.from_pretrained('distilgpt2')
trargs = TrainingArguments(".", do_train=True, num_train_epochs=5, per_device_train_batch_size=1, logging_steps=1, learning_rate=0.0001,
            per_device_eval_batch_size=1, eval_strategy="steps", eval_steps=1, report_to="none")
tr = Trainer(model=model, args=trargs, train_dataset=d, eval_dataset=dval, processing_class=t, data_collator=dc)
tr.train()

给定上面的约束，为了看到数据稀缺的影响，你应该做什么实验？
